In [41]:

# graph (Dictionary of Dictionaries)

graph = {
    'S': {'A': 3, 'B': 6, 'C': 5},
    'A': {'D': 9, 'E': 8},
    'B': {'F': 12, 'G': 14},
    'C': {'H': 7},
    'H': {'I': 5, 'J': 6},
    'I': {'K': 1, 'L': 10, 'M': 2},
    'D': {}, 'E': {}, 'F': {}, 'G': {},
    'J': {}, 'K': {}, 'L': {}, 'M': {}
}

heuristic = {
    'S': 10, 'A': 9, 'B': 7, 'C': 5, 'D': 8, 'E': 6, 'F': 4, 'G': 3,
    'H': 3, 'I': 2, 'J': 6, 'K': 2, 'L': 0, 'M': 1
}

# start and goal (Strings)

start_node = 'S'
goal_node = 'G'

# heuristic (Dictionary)

# heuristic = {
#     'Start': 8,
#     'A': 5,
#     'B': 6,
#     'C': 2,
#     'D': 7,
#     'Goal': 0
# }

# frontier (List of Tuples)
# What it does: The "waiting list" of nodes to explore next.
# Each item is a tuple containing (node_name, priority_score).
# *Note: In real-world apps, use Python's `heapq` module instead of a list.

# frontier = [('C', 4), ('B', 7), ('D', 15)]

# visited (Set)
# What it does: Keeps track of nodes that have already been completely explored.
# Why a Set?: Optimized for "membership testing" (O(1) lookup time)
# to prevent infinite loops if your graph has cycles.

# visited = {'Start', 'A'}

# cost_so_far (Dictionary)
# What it does: Used in UCS and A*. Represents g(n).
cost_so_far = {
    # 'Start': 0,
    # 'A': 5,
    # 'B': 2
}


# ---------------------------------------------------------------------
# 3. THE BREADCRUMB TRAIL (Reconstructing the Answer)
# ---------------------------------------------------------------------

# came_from (Dictionary)
# What it does: Maps a node to its "parent" (the node that immediately
# preceded it on the best path). When the algorithm hits the goal,
# it loops backward through this dictionary until it hits the start node.
came_from = {
    # 'Start': None,
    # 'A': 'Start',
    # 'B': 'Start',
    # 'C': 'A',
    # 'Goal': 'C'
}

# path (List)
# final_path = ['Start', 'A', 'C', 'Goal']

# helper functions for A* ucs bfs

In [42]:
import heapq

# HELPER 1: Path Builder (Used by all 3)
# Traces the breadcrumb trail backward and flips it.
def build_path(curr, came_from):
    path = []
    while curr: path.append(curr); curr = came_from[curr]
    return path[::-1]

# HELPER 2: Neighbor Evaluator for UCS & A*
# Calculates g(n), updates dictionaries if cheaper, and returns valid next steps.
def get_cheaper_neighbors(curr, graph, costs, came_from):
    valid_neighbors = []
    for nxt, edge_cost in graph.get(curr, {}).items():
        new_cost = costs[curr] + edge_cost
        if new_cost < costs.get(nxt, float('inf')):
            costs[nxt] = new_cost
            came_from[nxt] = curr
            valid_neighbors.append((new_cost, nxt))
    return valid_neighbors

# HELPER 3: Neighbor Evaluator for Greedy BFS
# Ignores g(n) entirely, only checks if a node is unvisited.
def get_unvisited_neighbors(curr, graph, came_from):
    valid_neighbors = []
    for nxt in graph.get(curr, {}):
        if nxt not in came_from:
            came_from[nxt] = curr
            valid_neighbors.append(nxt)
    return valid_neighbors

## BFS

In [43]:
def greedy_bfs(graph, start, goal, h):
    frontier = [(h[start], start)]
    came_from = {start: None}

    while frontier:
        _, curr = heapq.heappop(frontier)

        if curr == goal: return build_path(curr, came_from)

        for nxt in get_unvisited_neighbors(curr, graph, came_from):
            # Priority = h(n)
            heapq.heappush(frontier, (h[nxt], nxt))

    return None

## hill climbing

In [ ]:
def hill_climbing(graph, start, goal, h):
    curr = start
    came_from = {start: None}

    while curr:
        if curr == goal:
            return build_path(curr, came_from)

        # 1. Look at all immediate neighbors
        neighbors = list(graph.get(curr, {}).keys())

        if not neighbors:
            print("Hit a dead end. No path found.")
            return None

        # 2. Pick the absolute best neighbor based purely on heuristic
        best_nxt = min(neighbors, key=lambda n: h[n])

        # 3. The Local Maximum Check (The core Hill Climbing logic)
        if h[best_nxt] >= h[curr]:
            print("Stuck at a local maximum! Cannot move closer to goal.")
            return None

        # 4. Move forward
        # (Notice we don't save the other neighbors to a frontier. They are deleted forever!)
        came_from[best_nxt] = curr
        curr = best_nxt

    return None

## UCS

In [44]:
def ucs(graph, start, goal):
    frontier = [(0, start)]
    came_from = {start: None}
    costs = {start: 0}

    while frontier:
        _, curr = heapq.heappop(frontier)

        if curr == goal: return build_path(curr, came_from)

        for new_cost, nxt in get_cheaper_neighbors(curr, graph, costs, came_from):
            # Priority = g(n)
            heapq.heappush(frontier, (new_cost, nxt))

    return None

## Beam search

In [ ]:
def beam_search_ucs(graph, start, goal, k):
    frontier = [(0, start)] # Priority is g(n)
    came_from = {start: None}
    costs = {start: 0}

    while frontier:
        next_frontier = []

        # 1. Expand ALL nodes currently in the beam
        for current_cost, curr in frontier:

            if curr == goal:
                return build_path(curr, came_from)

            # 2. Use our helper to gather all valid neighbors
            neighbors = get_cheaper_neighbors(curr, graph, costs, came_from)
            next_frontier.extend(neighbors)

        # 3. The Beam Cut: Sort the massive list of new neighbors and keep only top 'k'
        next_frontier.sort(key=lambda x: x[0])
        frontier = next_frontier[:k]

    return None

## A*

In [45]:
def a_star(graph, start, goal, h):
    frontier = [(h[start], start)]
    came_from = {start: None}
    costs = {start: 0}

    while frontier:
        _, curr = heapq.heappop(frontier)

        if curr == goal: return build_path(curr, came_from)

        for new_cost, nxt in get_cheaper_neighbors(curr, graph, costs, came_from):
            # Priority = g(n) + h(n)
            heapq.heappush(frontier, (new_cost + h[nxt], nxt))

    return None

In [46]:
print(greedy_bfs(graph,start_node,goal_node,heuristic))
print(ucs(graph,start_node,goal_node))
print(a_star(graph,start_node,goal_node,heuristic))

['S', 'B', 'G']
['S', 'B', 'G']
['S', 'B', 'G']


## combination of UCS BFS A*

In [ ]:
import heapq

def master_search(graph, start, goal, h):
    # ==========================================
    # 1. INITIALIZE THE FRONTIER & TRACKERS
    # ==========================================
    # frontier = [(h[start], start)]   # BFS A*
    # frontier = [(0, start)]          # UCS
    came_from = {start: None}          # ALL
    # costs = {start: 0}               # UCS A*

    # ==========================================
    # 2. THE CORE ENGINE (Runs for all 3)
    # ==========================================
    while frontier:                    # ALL
        _, curr = heapq.heappop(frontier) # ALL

        if curr == goal:               # ALL
            return build_path(curr, came_from) # ALL (HELPER 1)

        # ==========================================
        # 3. EVALUATE NEIGHBORS
        # ==========================================

        # --- GREEDY BFS LOGIC --- #
        # for nxt in get_unvisited_neighbors(curr, graph, came_from):        # BFS (HELPER 3)
            # heapq.heappush(frontier, (h[nxt], nxt))                        # BFS

        # --- UCS & A* LOGIC --- #
        # for new_cost, nxt in get_cheaper_neighbors(curr, graph, costs, came_from): # UCS A* (HELPER 2)
            # heapq.heappush(frontier, (new_cost, nxt))                      # UCS
            # heapq.heappush(frontier, (new_cost + h[nxt], nxt))             # A*

    return None                        # ALL

## GA

In [ ]:
import random

# ==========================================
# 1. THE GENETIC OPERATORS (Helper Functions)
# ==========================================

def evaluate(chromosome):
    """Calculates fitness (Goal: maximize number of 1s)"""
    return sum(chromosome)

def select_parents(population, fitnesses):
    """Roulette Wheel Selection: higher fitness = higher chance to be picked"""
    # Python's random.choices does exact roulette wheel logic when given 'weights'!
    return random.choices(population, weights=fitnesses, k=2)

def crossover(p1, p2):
    """Single-point crossover: slices and swaps parent genes"""
    pt = random.randint(1, len(p1) - 1)
    return p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]

def mutate(chromosome, rate):
    """Bit-flip mutation: randomly flips 0s to 1s and vice versa"""
    return [(1 - bit if random.random() < rate else bit) for bit in chromosome]


# ==========================================
# 2. THE CORE ENGINE
# ==========================================

def genetic_algorithm(population, generations, mutation_rate):

    for _ in range(generations):
        next_generation = []

        # --- 1. EVALUATE ---
        # Score the entire current population
        fitnesses = [evaluate(chrom) for chrom in population]

        # --- 2. BREED ---
        # Loop enough times to fill the new generation (stepping by 2)
        for _ in range(len(population) // 2):

            # A. SELECT parents based on their scores
            p1, p2 = select_parents(population, fitnesses)

            # B. CROSSOVER to create two new children
            c1, c2 = crossover(p1, p2)

            # C. MUTATE the children and add them to the next generation pool
            next_generation.extend([mutate(c1, mutation_rate), mutate(c2, mutation_rate)])

        # --- 3. REPLACE ---
        # The old generation dies, the new generation takes over
        population = next_generation

    # Return the absolute best chromosome from the final generation
    return max(population, key=evaluate)


# ==========================================
# 3. EXECUTION
# ==========================================

initial_population = [
    [0, 1, 1, 0, 1],
    [1, 1, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 1, 1]
]

best_solution = genetic_algorithm(initial_population, generations=50, mutation_rate=0.01)

print("Best solution:", best_solution)
print("Fitness:", evaluate(best_solution))